In [ ]:
# Permite importar src/ desde la carpeta notebooks/
import sys
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import DIR_DATA_RAW

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid")

In [ ]:
from src.config import DIR_DATA_PROC
from src.calas import estimar_ventana_calas
from src.etiquetas import ventanas_finales

viajes = pd.read_parquet(DIR_DATA_PROC / "viajes.parquet")
calas = pd.read_parquet(DIR_DATA_PROC / "calas.parquet")
tel_v = pd.read_parquet(DIR_DATA_PROC / "telemetria_viajes.parquet")

# Solo viajes con arribo real y embarcaciones con telemetría
v_ok = viajes[(viajes["FuenteFin"] == "real") & viajes["Embarcacion"].isin(tel_v["ep"].unique())]
c_ok = calas[calas["IdViaje"].isin(v_ok["IdViaje"])]
print(f"Calas a ubicar: {len(c_ok):,} en {len(v_ok):,} viajes")

# 1. Perfil de velocidad alrededor de la hora registrada (como antes)
c2 = c_ok.merge(v_ok[["IdViaje", "Embarcacion"]], on="IdViaje")
filas = []
for ep, g in c2.groupby("Embarcacion"):
    t = tel_v.loc[tel_v["ep"] == ep, ["fecha", "Velocidad", "rpm", "gal_h"]]
    for h in g["HoraRegistrada"]:
        w = t[t["fecha"].between(h - pd.Timedelta(hours=3), h + pd.Timedelta(hours=3))].copy()
        w["min_rel"] = ((w["fecha"] - h).dt.total_seconds() / 60 / 15).round() * 15
        filas.append(w)
perfil = pd.concat(filas).groupby("min_rel")[["Velocidad", "rpm", "gal_h"]].median()
perfil.plot(subplots=True, figsize=(12, 7), marker="o", ms=3, title="Alrededor de la hora registrada")
plt.tight_layout()

# 2. Ventana real de cada cala
vent = estimar_ventana_calas(tel_v, c_ok, v_ok)
vent = vent.merge(c_ok[["IdViaje", "NumCala", "FuenteCala"]], on=["IdViaje", "NumCala"], how="left")
print(f"Calas detectadas: {vent['CalaDetectada'].mean():.1%}")
display(vent.groupby("FuenteCala")["CalaDetectada"].mean().round(3))
display(vent[["DuracionEstMin", "DesfaseInicioMin", "DesfaseFinMin"]].describe(percentiles=[.05, .25, .5, .75, .95]).round(1))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["DuracionEstMin", "DesfaseInicioMin", "DesfaseFinMin"]):
    vent[col].dropna().clip(-240, 240).hist(bins=32, ax=ax, color="#0065AC")
    ax.set_title(col)
plt.tight_layout()

# 3. Ventanas definitivas
vf = ventanas_finales(vent)
display(vf["MetodoVentana"].value_counts(normalize=True).round(3))
vf.to_parquet(DIR_DATA_PROC / "ventanas_calas.parquet", index=False)